# LearnKG: Reusable Positive-Only Prerequisite Evaluation (PRAT & Geometry)

This notebook runs the positive-only prerequisite evaluation for **PRAT** (*Networking & Internet*) and **CK-12 Geometry** using the unified evaluation framework.

### Evaluation Principles:
- **Positive-Only Ground Truth**: Evaluates against known positive prerequisite pairs (`PREREQUISITE -> TARGET`).
- **No Synthetic Negatives**: Zero unannotated pairs are assumed to be false negatives. Specificity and ROC-AUC are intentionally omitted.
- **Separation**: Datasets, outputs, and reports remain strictly decoupled.

## Step 1: Environment Setup & Repository Cloning
Prepares the environment, clones the repository (if in Colab), and verifies input paths.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

if IN_COLAB:
    if not Path('/content/LearnKG').exists():
        print('[*] Cloning LearnKG repository...')
        !git clone -b ACL https://github.com/thegreatvoid/LearnKG.git /content/LearnKG
    %cd /content/LearnKG

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

assert Path('data_input/PRAT/rawText.txt').exists(), 'Missing PRAT rawText.txt!'
assert Path('data_input/PRAT/GOLD-PR_PairsOnly.tsv').exists(), 'Missing PRAT GOLD-PR_PairsOnly.tsv!'
print('[+] Repository and PRAT dataset verified successfully. Working dir:', Path.cwd())

## Step 2: Install Core Dependencies
Install dependencies for parsing, graph manipulation, and evaluation.

In [ ]:
!pip install -q pandas numpy networkx scipy pyvis pypdf

## Step 3: Run Extraction Pipeline on PRAT
Extracts concepts and relationships from `data_input/PRAT/rawText.txt` into `data_output/PRAT/`.

In [ ]:
from educational_pipeline import run_educational_pipeline

print('[*] Running LearnKG educational pipeline on PRAT raw textbook text...')
concepts_csv, relations_csv, G = run_educational_pipeline(
    data_dir='PRAT',
    use_llm=False,
    use_neural_model=False,
    output_html='./docs/prat.html',
    source_filename='rawText.txt',
)
print(f'[+] Pipeline finished. Extracted concepts: {G.number_of_nodes()}, relations: {G.number_of_edges()}')

## Step 4: Run Unified Positive-Only Evaluation
Evaluates both PRAT and Geometry using the exact same evaluation function.

In [ ]:
from kg_eval.evaluate_positive_only import evaluate_dataset

# 1. Evaluate PRAT
print('[*] Evaluating PRAT on positive-only benchmark...')
report_prat = evaluate_dataset('prat')

print('=' * 65)
print('      PRAT POSITIVE-ONLY PREREQUISITE EVALUATION REPORT')
print('=' * 65)
print(f"Gold Concepts Total      : {report_prat['gold_concepts_total']}")
print(f"Gold Concepts Mapped     : {report_prat['gold_concepts_mapped']} ({report_prat['gold_concept_mapping_coverage']*100:.1f}%)")
print(f"Gold Pairs Total         : {report_prat['gold_positive_pairs_total']}")
print(f"Gold Pairs Evaluable     : {report_prat['gold_pairs_evaluable_subset']} ({report_prat['gold_pair_evaluable_coverage']*100:.1f}%)")
print(f"Predicted Prereq Pairs   : {report_prat['predicted_prerequisite_pairs_unique']}")
print('-' * 65)
print('EVALUABLE SUBSET (Pairs where both concepts mapped):')
for k, v in report_prat['metrics_evaluable_subset'].items():
    print(f"  {k:<25}: {v}")
print('-' * 65)
print(f"FULL GOLD SET (All {report_prat['gold_positive_pairs_total']} unique gold pairs):")
for k, v in report_prat['metrics_full_gold_set'].items():
    print(f"  {k:<25}: {v}")
print('=' * 65)

# 2. Reusability demonstration: Evaluate CK-12 Geometry with same function
print('\n[*] Reusability test: Evaluating CK-12 Geometry with the same evaluator...')
report_geom = evaluate_dataset('geometry')
print(f"[+] Geometry Gold Pairs: {report_geom['gold_positive_pairs_total']}, Predicted Prereqs: {report_geom['predicted_prerequisite_pairs_unique']}, Evaluable TP: {report_geom['metrics_evaluable_subset']['TP']}")
print(f"[+] Geometry Evaluable Precision: {report_geom['metrics_evaluable_subset']['Precision']}, Recall: {report_geom['metrics_evaluable_subset']['Recall']}")

## Step 5: Inspection of Samples & Generated Reports
Review missed gold pairs, unmatched predictions, and verify saved reports.

In [ ]:
print('\n--- PRAT: Sample Missed Gold Pairs (First 5) ---')
for item in report_prat['sample_missed_gold_pairs'][:5]:
    print(f"  [Gold PR] {item['prerequisite']} -> {item['target']} (Both mapped: {item['both_concepts_mapped']})")

print('\n--- PRAT: Sample Unmatched Predictions ---')
if report_prat['sample_unmatched_predictions']:
    for item in report_prat['sample_unmatched_predictions'][:5]:
        print(f"  [Pred] {item['source']} -> {item['target']}")
else:
    print('  None (0 predicted prerequisite pairs extracted by deterministic baseline)')

print('\n--- Reports Generated ---')
print('PRAT JSON Report    :', Path('data_output/PRAT/positive_only_evaluation_report.json').resolve())
print('PRAT Markdown Report:', Path('data_output/PRAT/positive_only_evaluation_report.md').resolve())
print('Geometry JSON Report:', Path('data_output/ck12_geometry/positive_only_evaluation_report.json').resolve())